# EXIST 2026 — Task 3 (vídeos) — notebook de orquestração

Este notebook **não define lógica**. Tudo — constantes, dataset, modelo, perdas,
métricas, IO de checkpoint — vive em `video_core.py`, mesmo padrão já aceito no projeto
com `EXIST_MEMES/memes_core.py`. Aqui só há orquestração: carregar dados, aplicar o
split congelado, ajustar o scaler, instanciar, treinar, avaliar, salvar.

Substitui `Thesis_video_fixed.ipynb`, que fica **intocado** como registo do sistema
submetido. As correções aplicadas estão especificadas em
`improvements/04_RECONCILIATION.md` e rastreadas em
`improvements/05_IMPLEMENTATION_NOTES.md`.

## O que mudou, em uma linha cada

| # | Mudança |
|---|---|
| ARCH-01 | `q_bias`/`v_bias` do VideoMAE restaurados do checkpoint (o `transformers` v5 não os mapeia; rodavam zerados nas 12 camadas) |
| ARCH-03 (b) | cross-attention **removida** — atendia sobre uma única key (softmax ≡ 1,0), era um mapa linear e consumia ≈ 45 % dos params treináveis |
| ARCH-05 | `max_length` 128 → **256** (128 truncava 42,9 % das transcrições) |
| ARCH-06 / ARCH-16 | **SWA removido**; nº de épocas é PROVISÓRIO até a run de calibração |
| ARCH-07 | modality dropout **por amostra**, faixas fixas e posicionais, taxa por modalidade idêntica em todos os braços |
| ARCH-08 | `bio_ln` removido (desfazia o `RobustScaler`); sem `BatchNorm1d` |
| ARCH-10 | formato hard da Task C não emite mais `NO` junto com categorias |
| ARCH-11 (b) | limiares da Task C ajustados numa metade do val e reportados na outra |
| ARCH-12 / OPT-03 | um `RUN_DIR` por (condição × seed); checkpoint enxuto (~44 MB, não 2,6 GB) |
| ARCH-14 | métricas que falham voltam `nan` em vez de sumirem da tabela; + proxies sklearn |
| ARCH-20 / OPT-13 | flags de ablação no **construtor**: cada condição é um modelo treinado do zero, e B0 não carrega slot bio (fusão 2560, não 2688) |
| OPT-06 / OPT-11 | `SPLIT_SEED` fixo separado de `INIT_SEED`; split 80/20 **congelado em disco** |
| DIV-06 / DIV-07 | semântica de `reaction_time` unificada; `LABELS_2_1` (que mentia a ordem) removida |

## O que este notebook **invalida**

Todos os checkpoints, CSVs de ablação, JSONs de limiares e predições existentes.
Ver a seção "O que este código invalida" em `improvements/05_IMPLEMENTATION_NOTES.md`.

## 1. Imports

Nada de lógica aqui — só o módulo e o mínimo para orquestrar.

In [1]:
# P1-fix: os caminhos deste notebook sao RELATIVOS a EXIST_VIDEOS/ (plano 11 §3:
# "kernel do Jupyter em /home/a67606/Thesis/EXIST_VIDEOS ou cd ate la"). Se o kernel
# nascer noutro cwd, o assert do JSON da celula seguinte falha. Idempotente.
import os
from pathlib import Path

_TARGET = Path("/home/a67606/Thesis/EXIST_VIDEOS")
if Path.cwd() != _TARGET:
    assert _TARGET.is_dir(), f"diretorio do projeto nao encontrado: {_TARGET}"
    os.chdir(_TARGET)
assert Path("Dataset").is_dir(), f"cwd errado: {Path.cwd()} (sem Dataset/)"
print("cwd:", Path.cwd())


cwd: /home/a67606/Thesis/EXIST_VIDEOS


In [2]:
import json

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import RobustScaler
from transformers import TrainingArguments, XLMRobertaTokenizer

import video_core as vc

## 2. Configuração da run

**Uma run = uma condição × uma `INIT_SEED`.** A tabela de ablação são 9 runs
(3 condições × 3 seeds), cada uma com o seu próprio checkpoint — os braços **não**
partilham pesos (era o ARCH-20).

- `SPLIT_SEED` é **fixo** e vive em `video_core`. Nunca muda, nunca é re-sorteado.
  Re-sortear porque os números saíram feios é *split-shopping*.
- `INIT_SEED` varia em {42, 7, 2025}: init dos pesos novos + ordem dos dados.
  Com o split fixo, o delta entre condições **dentro de uma seed** é pareado — que é a
  única análise válida (delta por seed + consistência de sinal, nunca `|Δmédia| > std`).

In [3]:
INIT_SEED      = 42        # ∈ {42, 7, 2025}
USE_BIO        = True      # False → B0_content_only
USE_VIEWER_EMB = True      # False → B1_bio_no_viewer ; True → B2_bio_viewer

CONDITION = vc.condition_tag(USE_BIO, USE_VIEWER_EMB)

# NUM_EPOCHS, RUN_DIR e CKPT_DIR são definidos na seção 2.1 (braço do P7), que é a
# ÚNICA fonte deles — dependem do braço. Não redefinir aqui: fonte dupla é como se
# roda um braço com o orçamento provisório de 6 épocas sem perceber.
print(f"condição = {CONDITION} | INIT_SEED = {INIT_SEED} | SPLIT_SEED = {vc.SPLIT_SEED}")
print(f"max_length = {vc.MAX_LENGTH}")


condição = B2_bio_viewer | INIT_SEED = 42 | SPLIT_SEED = 42
max_length = 256


## 2.1 Braço do P7 (atribuição em 3 braços)

Decisão do autor (plano 11 §13 Q5). `build_model` aceita `restore_biases`, então o
ARCH-01 isola-se pelo lado do MODELO, sem extração extra:

| braço | frames | áudio | `restore_biases` | delta isola |
|---|---|---|---|---|
| baseline (congelado) | v1 | v1 | código antigo | referência |
| **A′** | v2 | v1 | `False` | frames + código (**bloco único**) |
| **A** | v2 | v1 | `True` | **ARCH-01** = A − A′ |
| **B** | v2 | **v2** | `True` | **áudio** = B − A |

`B − baseline` = ganho total; os três deltas o decompõem.

**Protocolo de épocas (obrigatório — regra de simetria, `09_BASELINE_RESULTS.md` §2):**
uma calibração de 15 épocas na configuração final (B) → congelar `E = argmin(eval_loss)`
→ rodar os TRÊS braços com esse mesmo E. Estado final contra estado final, sempre.

⚠ Frames são **sempre v2** nos quatro: a árvore v1 foi apagada (V6) e não é regenerável.
Por isso A′ mistura correções de extração de frames com demais correções de código e tem
de ser reportado como bloco único, sem proporção interna.


In [4]:
from pathlib import Path

ARM = "B15"   # "CAL" | "A_prime" | "A" | "B" | "B15"  ← ÚNICA LINHA A EDITAR

# Frames são SEMPRE v2 (a árvore v1 não existe mais). O que varia por braço é a
# geração das features de ÁUDIO e o flag restore_biases do build_model.
_ARMS = {              # ARM: (audio_version, restore_biases, epochs|None)
    "CAL":     ("v2", True,  vc.CALIBRATION_EPOCHS),  # calibração: config final (= B)
    "A_prime": ("v1", False, None),   # frames v2 + código, sem ARCH-01
    "A":       ("v1", True,  None),   # A − A′  = ARCH-01
    "B":       ("v2", True,  None),   # B − A   = áudio (decomposição, E congelado)
    # B15: mesma config do B, mas com o ORÇAMENTO DE ÉPOCAS DO BASELINE (15).
    # Existe porque `B − baseline` com E=5 é confundido: o baseline é o SWA de uma
    # run de 15 épocas. Com 15 aqui, a comparação mede features+código, não orçamento.
    # ⚠ Resta uma diferença DECLARADA: o baseline usa SWA, este não (ARCH-06).
    "B15":     ("v2", True,  15),
}
assert ARM in _ARMS, f"ARM inválido: {ARM!r}"
AUDIO_VERSION, RESTORE_BIASES, _ARM_EPOCHS = _ARMS[ARM]

# E = argmin(eval_loss) da run de CALIBRAÇÃO. Preencher DEPOIS de ler a curva da
# célula 12 na run CAL; os três braços rodam com este mesmo valor.
FROZEN_EPOCHS = 5

NUM_EPOCHS = _ARM_EPOCHS if _ARM_EPOCHS is not None else FROZEN_EPOCHS
assert NUM_EPOCHS is not None, (
    "FROZEN_EPOCHS ainda é None: rodar ARM='CAL' primeiro, ler E = argmin(eval_loss) "
    "na curva de validação e congelar aqui. Sem isso os braços não são comparáveis.")

# Sufixo por braço: os três compartilham a condição (bio+viewer), então sem ele as
# runs se sobrescreveriam. Derivado aqui para não alterar video_core.run_dir/ckpt_dir.
RUN_DIR  = Path(f"{vc.run_dir(INIT_SEED, USE_BIO, USE_VIEWER_EMB)}_{ARM}")
CKPT_DIR = Path(f"{vc.ckpt_dir(INIT_SEED, USE_BIO, USE_VIEWER_EMB)}_{ARM}")

print(f"ARM = {ARM} | áudio = {AUDIO_VERSION} | frames = v2 | "
      f"restore_biases = {RESTORE_BIASES}")
_why = {"CAL": " (CALIBRAÇÃO)", "B15": " (orçamento do baseline)"}.get(ARM, " (E congelado)")
print(f"épocas = {NUM_EPOCHS}{_why}")
print(f"RUN_DIR  = {RUN_DIR}")
print(f"CKPT_DIR = {CKPT_DIR}")


ARM = B15 | áudio = v2 | frames = v2 | restore_biases = True
épocas = 15 (orçamento do baseline)
RUN_DIR  = run_B2_bio_viewer_seed42_B15
CKPT_DIR = checkpoints_B2_bio_viewer_seed42_B15


## 3. Determinismo e self-checks

`set_all_seeds` é chamado **de novo imediatamente antes de cada init de pesos** (é o que
`vc.build_model` faz internamente) — as matrizes LoRA A são aleatórias, e o
`set_seed(42)` interno do Trainer chega tarde demais.

Os self-checks custam milissegundos e travam três coisas que falham em silêncio:
a convenção da Task A (invertida em relação ao `memes_core.py`), a largura da fusão por
condição, e a taxa do modality dropout por modalidade.

In [5]:
vc.set_all_seeds(INIT_SEED)
vc.run_selfchecks()

✓ convenção Task A (sigmoid = P(YES), eixo [NO, YES]) verificada
✓ fusion_input_dim: 2688 com bio (B1/B2), 2560 sem bio (B0)
✓ modality dropout: taxas por modalidade ≈ ['0.0503', '0.0495', '0.0499', '0.0501'] (p=0.05); B0 apenas não usa a 4ª faixa, as outras três não mudam


## 4. Dados e split **congelado**

O split é um artefato em disco (`Dataset/val_split_42.json`), commitado no repositório.
Todas as runs carregam a mesma lista de `id_EXIST` em vez de recalcular — assim
"as 9 runs partilham o mesmo val" é **verificável**, não uma promessa.

Na primeira criação, o split passa pelo portão do §6.3: se alguma categoria da Task C
ficar com contagem de positivos de um dígito no val, ele re-estratifica pela chave
conjunta e, se ainda assim falhar, **recusa-se a gravar**. Já foi criado e passou na
primeira tentativa (estratificação por Task A, 2019/505, categoria mais rara com 31
positivos).

⚠ **O treino roda numa VM separada e `Dataset/` não é sincronizado.** Copiar `Dataset/val_split_42.json` para a VM antes da primeira run: a célula abaixo tem de imprimir `[split] carregado de ...`. Se imprimir `[split] artefato gravado`, a run está **fora de protocolo** — o split foi recalculado em vez de carregado.


In [6]:
# O artefato do split é COMMITADO no repositório, mas `Dataset/` não é sincronizado
# com a VM automaticamente. Sem este assert, `build_frozen_split` cairia no ramo de
# CRIAÇÃO e recalcularia o split em silêncio — as runs continuariam a partilhar um val,
# mas deixaria de ser O val congelado, e a garantia do §6.2 vira promessa outra vez.
assert vc.split_artifact_path().exists(), (
    "Copiar Dataset/val_split_42.json para a VM antes de rodar. Sem ele, "
    "build_frozen_split RECALCULA o split em vez de carregar o congelado.")

full_df = pd.read_json("./Dataset/training/EXIST2026_training.json").transpose()
train_df, val_df = vc.build_frozen_split(full_df)
print(f"Treino: {len(train_df)} | Val: {len(val_df)}")

[split] carregado de Dataset/val_split_42.json (seed=42, estratificação=task_a): 2019 treino / 505 val
Treino: 2019 | Val: 505


## 5. `RobustScaler` das features biométricas

Ajustado **só no treino** (anti-leakage) e gravado no `RUN_DIR`, junto com os pesos —
o `inference_video.py` tem de recebê-lo por `--scaler` (obrigatório: ajustar um scaler
novo no conjunto avaliado é vazamento, BUG-B/ARCH-18).

A contagem de imputações de `reaction_time` é impressa de propósito: **rodar esta mesma
contagem sobre o JSON de test antes de submeter** (DIV-06).

In [7]:
train_bio_matrix = vc.collect_bio_vectors(train_df)
scaler = RobustScaler().fit(train_bio_matrix)
print("matriz bio de treino:", train_bio_matrix.shape)

[bio] 4038 linhas utilizador-vídeo | reaction_time imputado em 77 (1.91 %)
matriz bio de treino: (4038, 16)


## 6. Datasets

`max_length=256` (ARCH-05). Tem de ser **o mesmo** em qualquer caminho de inferência.

In [8]:
tokenizer = XLMRobertaTokenizer.from_pretrained(vc.XLMR_NAME)

train_dataset = vc.EXISTVideoDataset(
    train_df, tokenizer, scaler,
    vc.frames_dir("train"), vc.audio_dir("train", version=AUDIO_VERSION), max_length=vc.MAX_LENGTH)
val_dataset = vc.EXISTVideoDataset(
    val_df, tokenizer, scaler,
    vc.frames_dir("train"), vc.audio_dir("train", version=AUDIO_VERSION), max_length=vc.MAX_LENGTH)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Train: 2019 | Val: 505


## 7. Sanity checks dos dados

In [9]:
sample = train_dataset[0]
for k, v in sample.items():
    print(f"{k:20s}: {tuple(v.shape) if torch.is_tensor(v) else v}")

assert sample["frames"].shape == (vc.NUM_FRAMES, vc.NUM_CHANNELS, vc.FRAME_H, vc.FRAME_W)
print("✓ shape dos frames confere")

# O batch NÃO tem nenhuma chave chamada "labels" — é o que mantém a divisão por
# gradient_accumulation_steps ativa (ARCH-02). MultiTaskTrainer trava a flag em False
# de qualquer forma, mas confirmar aqui é grátis.
print("'labels' no batch?", "labels" in sample, "(tem de ser False)")

cvpc_values = [train_dataset[i]["cvpc"].item() for i in range(len(train_dataset))]
print(f"CVPC mean={np.mean(cvpc_values):.4f} std={np.std(cvpc_values):.4f} "
      f"range=[{min(cvpc_values):.4f}, {max(cvpc_values):.4f}]")

id_EXIST            : 120001
frames              : (16, 3, 224, 224)
audio               : (1280,)
input_ids           : (256,)
attention_mask      : (256,)
bio_vec             : (16,)
cvpc                : (1,)
user_ids            : (2,)
labels_a            : ()
labels_b            : (3,)
labels_c            : (6,)
✓ shape dos frames confere
'labels' no batch? False (tem de ser False)
CVPC mean=0.0883 std=0.3861 range=[-0.8384, 1.0000]


## 8. Modelo

`vc.build_model` faz, na ordem do protocolo de determinismo:
re-seed → VideoMAE `from_pretrained` → **restaura `q_bias`/`v_bias`** (ARCH-01) →
LoRA → re-seed → XLM-R → LoRA → re-seed → `EXISTVideoModel` na condição pedida.

⚠ Fingerprints só são comparáveis **dentro do mesmo par (condição, INIT_SEED)**: braços
diferentes criam módulos diferentes e consomem RNG de forma diferente.

⚠ Confirmar a contagem de params por braço **antes** de qualquer número ir para a nota
de rodapé da tese — a tabela do §1.4 da reconciliação (B0 ≈ 5,88 M, B1/B2 ≈ 6,03 M) é
aritmética feita à mão.

In [10]:
model = vc.build_model(INIT_SEED, use_bio=USE_BIO,
                       use_viewer_emb=USE_VIEWER_EMB,
                       restore_biases=RESTORE_BIASES)

print("fusion_in:", model.fusion_in,
      "(2560 = B0 sem slot bio | 2688 = B1/B2)")
print("params:", vc.count_params(model))
vc.record_fingerprint(RUN_DIR, "init", model)

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
videomae.encoder.layer.{0...11}.attention.attention.v_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.q_bias       | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.bias                | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias      | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias           | UNEXPECTED

[fix ARCH-01] q_bias/v_bias restaurados em 12 camadas do VideoMAE


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fusion_in: 2688 (2560 = B0 sem slot bio | 2688 = B1/B2)
params: {'total': 652156891, 'trainable': 6030043, 'trainable_pct': 0.9246307266267927}
[fingerprint] init: ddb48bff571f413f  (primeira run; nada a comparar)


'ddb48bff571f413f'

## 9. `TrainingArguments`

**Esta célula tem quatro donos e é a de maior risco de alteração acidental no meio de
uma série.** Depois de congelada para a run de calibração, ela **não muda** até o fim da
tabela de ablação — qualquer alteração obriga a repetir a calibração.

Decisões registadas:

- `warmup_steps=0.1` (float `[0,1)` = ratio). Isto é **durabilidade, não bugfix**:
  `warmup_ratio` **funciona** na versão da VM (`transformers==5.6.0`) — o treino
  original rodou com ele, emitindo só um aviso de deprecação. Migrar prepara para uma
  versão futura. Confirmar equivalência imprimindo `get_warmup_steps(total_steps)`.
- `load_best_model_at_end=False` e `metric_for_best_model` **deliberadamente não
  definido**: selecionar a época pela mesma métrica reportada, sobre o mesmo split, é o
  vazamento nº 2 do precedente dos memes. Com a flag em False `metric_for_best_model`
  seria inerte, mas defini-lo arma o vazamento para quem virar a flag depois.
- **batch/checkpointing:** mantidos na configuração que **sabidamente roda**
  (micro-batch 2 × accum 16 = efetivo 32, com gradient checkpointing). O OPT-04 propõe
  bs=8 / accum=4 / sem checkpointing — mesmo batch efetivo, possivelmente 1,5–2× mais
  rápido — mas é `[MUDA]` (muda o consumo de RNG e os pesos bit a bit), depende de VRAM
  não medida e do OPT-01 (frames em fp16/uint8, que exige re-extração). **Decidir com o
  benchmark abaixo e congelar ANTES da run de calibração.**
- `save_strategy="no"`: o Trainer serializava 2,6 GB por época e **nenhum** desses
  checkpoints era lido. O modelo final é salvo à mão, enxuto, na seção 12.

In [11]:
training_args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=NUM_EPOCHS,

    # ── batch: configuração CONGELADA (a que sabidamente roda) ──────────────
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,          # batch efetivo = 32
    gradient_checkpointing=True,
    # Candidato do OPT-04, a validar no benchmark da Fase 0 e congelar antes da
    # calibração (mesmo batch efetivo, sem checkpointing):
    #     per_device_train_batch_size=8,
    #     gradient_accumulation_steps=4,
    #     per_device_eval_batch_size=16,
    #     gradient_checkpointing=False,

    bf16=True,
    fp16=False,

    dataloader_num_workers=2,                # OPT-05 sugere 8 — depende de `nproc` na VM
    dataloader_persistent_workers=True,      # evita recriar o pool a cada época/avaliação
    dataloader_pin_memory=True,

    eval_strategy="epoch",
    save_strategy="no",                      # OPT-03: nenhum checkpoint intermédio era lido
    load_best_model_at_end=vc.LOAD_BEST_MODEL_AT_END,   # False — não ligar (leakage)
    # metric_for_best_model: deliberadamente AUSENTE. Ver markdown acima.

    warmup_steps=0.1,                        # float ∈ [0,1) = ratio (durabilidade, não bugfix)
    lr_scheduler_type="cosine",

    remove_unused_columns=False,             # o dataset devolve campos que o forward
                                             # consome por nome (e `id_EXIST`, engolido
                                             # pelo **kwargs)
    label_names=["labels_a", "labels_b", "labels_c"],

    report_to="none",
    logging_steps=10,
    seed=INIT_SEED,
    data_seed=INIT_SEED,
)

### 9.1 Benchmark do batch (rodar UMA vez, na VM, antes de congelar)

Não é para rodar em toda run. É a medição que decide a configuração de batch; depois
dela, a escolha entra na célula acima e **fica congelada**.

In [12]:
# Descomentar para medir. ~15 min. Comparar train_samples_per_second e a memória de pico.
#
# for bs, accum, ckpt in [(2, 16, True), (2, 16, False), (8, 4, True), (8, 4, False)]:
#     torch.cuda.reset_peak_memory_stats()
#     bench_args = TrainingArguments(
#         output_dir="./bench_tmp", max_steps=30, per_device_train_batch_size=bs,
#         gradient_accumulation_steps=accum, gradient_checkpointing=ckpt,
#         bf16=True, save_strategy="no", eval_strategy="no",
#         remove_unused_columns=False, report_to="none",
#         seed=INIT_SEED, data_seed=INIT_SEED,
#     )
#     bench_model = vc.build_model(INIT_SEED, use_bio=USE_BIO, use_viewer_emb=USE_VIEWER_EMB)
#     bench = vc.MultiTaskTrainer(model=bench_model, args=bench_args,
#                                 train_dataset=train_dataset,
#                                 data_collator=vc.video_collate)
#     out = bench.train()
#     print(f"bs={bs} accum={accum} ckpt={ckpt}: "
#           f"{out.metrics['train_samples_per_second']:.2f} amostras/s | "
#           f"pico {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
#     del bench, bench_model; torch.cuda.empty_cache()

## 10. Trainer

`data_collator=vc.video_collate` é **obrigatório**: o dataset devolve `id_EXIST` (str) e
o `default_data_collator` do HF tentaria `torch.tensor` sobre uma lista de strings.

`trainer.label_names` é setado também depois da construção — é a forma empiricamente
conhecida por funcionar em `transformers==5.6.0`, e é o que faz `eval_pred.label_ids`
chegar a `compute_metrics` como uma 3-tupla. **Não remover.**

Sem `SWACallback` (ARCH-16) e sem `EarlyStoppingCallback`.

In [13]:
optimizer = vc.get_optimizer(model)

trainer = vc.MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=vc.compute_metrics,
    data_collator=vc.video_collate,
    optimizers=(optimizer, None),
)
trainer.label_names = ["labels_a", "labels_b", "labels_c"]

# Verificação do ARCH-02 e da equivalência warmup_ratio → warmup_steps.
print("model_accepts_loss_kwargs:", trainer.model_accepts_loss_kwargs, "(travado em False)")
total_steps = (len(train_dataset) // (training_args.per_device_train_batch_size
               * training_args.gradient_accumulation_steps)) * NUM_EPOCHS
print("warmup steps:", training_args.get_warmup_steps(total_steps),
      f"de ~{total_steps} steps totais")

model_accepts_loss_kwargs: False (travado em False)
warmup steps: 95 de ~945 steps totais


## 11. Treino

In [14]:
train_output = trainer.train()
print(train_output)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss,Task A Icm Hard,Task A Icmnorm Hard,Task A Fmeasure Hard,Task A Icmsoft Soft,Task A Icmsoftnorm Soft,Task A Crossentropy Soft,Task B Icm Hard,Task B Icmnorm Hard,Task B Fmeasure Hard,Task B Icmsoft Soft,Task B Icmsoftnorm Soft,Task B Crossentropy Soft,Task C Icmsoft Soft,Task C Icmsoftnorm Soft,Task C Icm Hard,Task C Icmnorm Hard,Task C Fmeasure Hard,Proxy Task A F1,Proxy Task A Auc,Proxy Task B F1,Proxy Task C F1
1,1.903381,3.193260,-0.222031,0.388959,0.590110,-1.267660,0.277248,1.033081,-0.702245,0.257765,nan,-3.060126,0.147111,nan,-10.166544,0.000000,-1.280163,0.190142,nan,0.590110,0.636281,0.373711,0.103981
2,1.976095,1.661671,-0.266034,0.366953,0.550893,-1.169895,0.294427,1.007293,-0.806335,0.221860,nan,-2.802107,0.176865,nan,-9.223946,0.018495,-1.326064,0.179032,nan,0.550893,0.631331,0.314675,0.113517
3,1.485908,1.436117,0.059313,0.529663,0.685053,-0.573487,0.399228,0.859444,-0.187849,0.435203,nan,-1.917873,0.278834,nan,-8.881219,0.036386,-0.885449,0.285681,nan,0.685053,0.752526,0.547198,0.193734
4,1.483276,1.419251,0.050570,0.525291,0.673344,-0.437603,0.423105,0.847418,-0.300285,0.396419,nan,-1.674349,0.306917,nan,-9.228627,0.018250,-0.768273,0.314043,nan,0.673344,0.785569,0.476005,0.196033
5,1.253744,1.344454,0.110906,0.555466,0.697740,-0.331586,0.441734,0.800565,-0.042303,0.485408,nan,-1.649295,0.309806,nan,-8.073831,0.078533,-0.745617,0.319527,nan,0.697740,0.802255,0.590391,0.210558
6,1.215011,1.379278,0.165061,0.582550,0.717800,-0.281775,0.450487,0.831339,0.040659,0.514025,nan,-1.333086,0.346270,nan,-8.338057,0.064740,-0.570204,0.361985,nan,0.717800,0.807009,0.621255,0.256770
7,1.086635,1.384564,0.205586,0.602816,0.733650,-0.170762,0.469994,0.837918,0.049104,0.516938,nan,-1.218636,0.359469,nan,-7.690362,0.098550,-0.494936,0.380203,nan,0.733650,0.804694,0.625256,0.318648
8,1.059876,1.406247,0.166370,0.583204,0.721109,-0.027388,0.495187,0.844221,-0.004125,0.498577,nan,-1.085963,0.374768,nan,-7.113171,0.128681,-0.592208,0.356659,nan,0.721109,0.804020,0.607466,0.307885
9,0.873345,1.475461,0.192785,0.596415,0.730514,-0.034334,0.493967,0.888662,-0.007596,0.497380,nan,-1.000159,0.384663,nan,-7.293807,0.119251,-0.533315,0.370913,nan,0.730514,0.795963,0.608848,0.346975
10,0.856921,1.557751,0.167897,0.583968,0.722684,0.026883,0.504724,0.945323,0.048318,0.516667,nan,-0.952429,0.390167,nan,-6.812742,0.144364,-0.503554,0.378117,nan,0.722684,0.790432,0.624846,0.364891


2026-08-09 17:47:35,754 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-08-09 17:47:35,807 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-09 17:47:35,930 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-08-09 17:47:35,932 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-09 17:47:36,052 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-09 17:47:36,168 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
2026-08-09 17:47:36,282 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']
2026-08-09 17:47:36,372 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft eva

In [15]:
# Por que FMeasure/CrossEntropy vêm NaN nas tasks B e C — registro para a tese.
import numpy as np, tempfile
from pyevall.evaluation import PyEvALLEvaluation
import video_core as vc

pred_out = trainer.predict(val_dataset)
_, logits_b, _ = vc._unpack_predictions(pred_out.predictions)
_, labels_b, _ = pred_out.label_ids
logits_b, labels_b = np.asarray(logits_b), np.asarray(labels_b)
mask = labels_b[:, 0] != vc.SENTINEL_BC
x = logits_b[mask]; x = x - x.max(axis=-1, keepdims=True)
probs = np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)
ids = [str(i) for i in range(len(probs))]

with tempfile.TemporaryDirectory() as td:
    ml = vc._metric_list('2_2', 'hard')
    preds = vc._format_for_pyevall(probs, ids, '2_2', 'hard')
    gold = vc._format_for_pyevall(labels_b[mask], ids, '2_2', 'hard')
    import json, os
    pp, gp = f"{td}/p.json", f"{td}/g.json"
    json.dump(preds, open(pp, "w")); json.dump(gold, open(gp, "w"))
    from pyevall.utils.utils import PyEvALLUtils
    rep = PyEvALLEvaluation().evaluate(
        pp, gp, ml, **{PyEvALLUtils.PARAM_HIERARCHY: vc.TASK_2_2_HIERARCHY})
    for m in ml:
        d = rep.report.get('metrics', {}).get(m)
        print(f"  {m}: {'AUSENTE do relatório' if d is None else 'status=' + str(d.get('status'))}")

2026-08-09 19:15:01,101 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-08-09 19:15:01,147 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-09 19:15:01,268 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-08-09 19:15:01,269 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-09 19:15:01,388 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-09 19:15:01,501 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
2026-08-09 19:15:01,620 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']
2026-08-09 19:15:01,705 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft eva

## 12. Curva de validação — leitura do orçamento de épocas

**Só relevante na run de calibração** (`NUM_EPOCHS = vc.CALIBRATION_EPOCHS`).

Critério fixado *a priori*, não negociável depois de ver a curva:

> **E = argmin(`eval_loss`)** sobre as 15 épocas, com `INIT_SEED=42` e o split
> congelado. Congelar E e usá-lo, sem alteração, em todas as condições e seeds.

Por que `eval_loss` e não a ICM: escolher a época pela **mesma métrica que se reporta** é
o vazamento nº 2 do precedente dos memes. `eval_loss` é um escalar único, independente
do PyEvALL, e não é o número que vai para a tabela — é o critério menos contaminado
disponível sem um terceiro split, e é conservador.

Declarar na tese, textualmente: *orçamento de épocas escolhido uma vez, no conjunto de
desenvolvimento, numa única seed, pelo mínimo da loss de validação, e congelado para
todas as condições e seeds.* Registar também as 6 métricas em E e em E±2, para
documentar a sensibilidade — **sem usar isso para mudar E**.

In [16]:
evals = [h for h in trainer.state.log_history if "eval_loss" in h]
if evals:
    curva = pd.DataFrame(evals)
    cols = [c for c in curva.columns
            if c in ("epoch", "eval_loss")
            or c.startswith(("eval_task_a", "eval_task_b", "eval_task_c"))]
    print(curva[cols].to_string(index=False))

    best = min(evals, key=lambda h: h["eval_loss"])
    print(f"\nE = argmin(eval_loss) = época {best['epoch']:.0f} "
          f"(eval_loss = {best['eval_loss']:.4f})")
    print("→ congelar este E em vc.NUM_TRAIN_EPOCHS_PROVISIONAL e NÃO mexer mais.")

 eval_loss  eval_task_a_icm_hard  eval_task_a_icmnorm_hard  eval_task_a_fmeasure_hard  eval_task_a_icmsoft_soft  eval_task_a_icmsoftnorm_soft  eval_task_a_crossentropy_soft  eval_task_b_icm_hard  eval_task_b_icmnorm_hard  eval_task_b_fmeasure_hard  eval_task_b_icmsoft_soft  eval_task_b_icmsoftnorm_soft  eval_task_b_crossentropy_soft  eval_task_c_icmsoft_soft  eval_task_c_icmsoftnorm_soft  eval_task_c_icm_hard  eval_task_c_icmnorm_hard  eval_task_c_fmeasure_hard  epoch
  3.193260             -0.222031                  0.388959                   0.590110                 -1.267660                      0.277248                       1.033081             -0.702245                  0.257765                        NaN                 -3.060126                      0.147111                            NaN                -10.166544                      0.000000             -1.280163                  0.190142                        NaN    1.0
  1.661671             -0.266034                  0.36

## 13. Salvar a run (checkpoint enxuto)

Um diretório por (condição × seed), com pesos treináveis + LoRA (~44 MB), scaler,
tokenizer, fingerprints e `meta.json`.

⚠ **ARCH-01 × OPT-03:** os `q_bias`/`v_bias` restaurados são *frozen* e **não** entram
neste arquivo. Quem carregar tem de reaplicar a restauração — é o que `vc.load_run` faz,
via `build_backbones`. Construir o modelo por outra via e carregar este arquivo produz
um modelo servido diferente do treinado, **sem aviso**.

In [17]:
from accelerate.utils import extract_model_from_parallel

actual_model = extract_model_from_parallel(trainer.model)
# `save_run` grava "videomae_qv_biases" com uma string FIXA que não olha o flag
# `restore_biases` — no braço A_prime isso seria falso. `full_meta.update(meta)`
# deixa sobrescrever, então a verdade do braço é registrada aqui.
vc.save_run(actual_model, RUN_DIR, scaler=scaler, tokenizer=tokenizer,
            meta={"init_seed": INIT_SEED, "condition": CONDITION,
                  "num_train_epochs": NUM_EPOCHS,
                  "train_runtime_s": train_output.metrics.get("train_runtime"),
                  "arm": ARM,
                  "restore_biases": bool(RESTORE_BIASES),
                  "audio_version": AUDIO_VERSION,
                  "frames_version": vc.FEATURES_VERSION,
                  "videomae_qv_biases": (
                      "restored_at_build_not_in_checkpoint" if RESTORE_BIASES
                      else "ZEROED_at_build (ARCH-01 desligado — braço A_prime)")})
vc.record_fingerprint(RUN_DIR, "final", actual_model)

[save_run] gravado em run_B2_bio_viewer_seed42_B15 (condição: use_bio=True, use_viewer_emb=True, fusion_in=2688)
[fingerprint] final: c322520695e01c81  (primeira run; nada a comparar)


'c322520695e01c81'

## 14. Validação do checkpoint

No sistema antigo esta célula **crashou** (gravava num diretório e lia de outro), então
nenhuma verificação de integridade jamais rodou. Agora lê exatamente o que a gravação
escreveu, e falha alto se as chaves não baterem.

In [18]:
model_check, scaler_check = vc.load_run(RUN_DIR, init_seed=INIT_SEED)

mismatches = 0
for key in ["fusion.0.weight", "head_a.weight", "head_b.weight", "head_c.weight"]:
    orig = actual_model.state_dict()[key].cpu()
    loaded = model_check.state_dict()[key].cpu()
    if torch.allclose(orig, loaded):
        print(f"  OK: {key}")
    else:
        print(f"  MISMATCH: {key}")
        mismatches += 1

fresh = vc.build_model(INIT_SEED, use_bio=USE_BIO, use_viewer_emb=USE_VIEWER_EMB)
differs_from_fresh = not torch.allclose(
    model_check.state_dict()["head_a.weight"].cpu(),
    fresh.state_dict()["head_a.weight"].cpu())

if mismatches == 0 and differs_from_fresh:
    print("\n✓ Checkpoint válido — bate com o modelo treinado e difere do init aleatório")
elif mismatches:
    print(f"\n✗ {mismatches} divergências de peso — algo correu mal no salvamento")
else:
    print("\n✗ Os pesos carregados são idênticos ao init aleatório — não foi salvo")

del model_check, fresh

2026-08-09 19:15:11,757 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/MCG-NJU/videomae-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-09 19:15:11,779 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/MCG-NJU/videomae-base/dc740ceda42fce44faed2ea03c6d447db72f6af9/config.json "HTTP/1.1 200 OK"
2026-08-09 19:15:11,912 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/MCG-NJU/videomae-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-09 19:15:11,933 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/MCG-NJU/videomae-base/dc740ceda42fce44faed2ea03c6d447db72f6af9/config.json "HTTP/1.1 200 OK"
2026-08-09 19:15:12,208 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/MCG-NJU/videomae-base/resolve/main/model.safetensors "HTTP/1.1

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

[transformers] VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
videomae.encoder.layer.{0...11}.attention.attention.v_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.q_bias       | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.bias                | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.value.weight | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias      | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias           | UNEXPECTED

2026-08-09 19:15:14,138 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/MCG-NJU/videomae-base/resolve/main/model.safetensors "HTTP/1.1 302 Found"
[fix ARCH-01] q_bias/v_bias restaurados em 12 camadas do VideoMAE
2026-08-09 19:15:14,315 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-08-09 19:15:14,447 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-08-09 19:15:14,592 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (3 outputs left out)

2026-08-09 19:15:17,865 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/MCG-NJU/videomae-base/resolve/main/model.safetensors "HTTP/1.1 302 Found"
[fix ARCH-01] q_bias/v_bias restaurados em 12 camadas do VideoMAE
2026-08-09 19:15:18,038 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-08-09 19:15:18,169 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
2026-08-09 19:15:18,311 - httpx - INFO - _send_single_request() - HTTP Request: HEAD https://huggingface.co/xlm-roberta-large/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✓ Checkpoint válido — bate com o modelo treinado e difere do init aleatório


## 15. Ajuste de limiares da Task C (uma vez, pós-treino)

**Único** lugar onde a busca de limiares roda. Rodar por época ajustaria os limiares ao
ruído do dev set.

Os limiares são ajustados numa metade do val e o número reportável vem da **outra**
metade (ARCH-11 b). `NONE` fica fixo em 0,5 — com a regra hard do ARCH-10 o seu limiar
só entra em jogo quando nenhuma categoria bate a sua, e deixá-lo livre era um grau de
liberdade que só produzia predições contraditórias.

⚠ **Assimetria a declarar na tese:** o Task C hard reportado aqui vem de ~252 amostras,
enquanto Task A/B (da célula de avaliação) vêm das ~505 do val inteiro.

In [19]:
threshold_result = vc.tune_task_c_thresholds(
    trainer, val_dataset,
    save_path=str(RUN_DIR / "task_c_thresholds.json"),
)

[Threshold tuning] Gerando predições no conjunto de validação...


2026-08-09 19:15:42,174 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-08-09 19:15:42,219 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-09 19:15:42,340 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-08-09 19:15:42,342 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-09 19:15:42,461 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-09 19:15:42,575 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
2026-08-09 19:15:42,689 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']
2026-08-09 19:15:42,780 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft eva

## 16. Avaliação final e próximos passos

`trainer.evaluate()` é o **caminho único** de avaliação (OPT-08): com as flags de
ablação no construtor, cada condição é um modelo, e não há mais um loop de inferência
paralelo no harness produzindo números num regime numérico diferente (era o BUG-H:
0,3531 do `evaluate()` contra 0,34764 do harness, mesmo modelo, mesma seed).

**Depois das 9 runs:**

1. Agregar com **delta pareado por seed + consistência de sinal** — nunca
   `|Δmédia| > std`. Para cada métrica e cada par (condição X, baseline B):
   `δ_s = m(X, s) − m(B, s)` para `s ∈ {42, 7, 2025}`; reportar os três δ, a mediana e
   se o sinal é consistente.
2. `probe_zero_video` (04_RECONCILIATION §2.3) — **análise separada, nunca na tabela de
   ablação**, e com leitura assimétrica: se as métricas não caem, isso fecha a
   cross-attention e é um achado negativo publicável; se caem, **não é resultado
   nenhum**, é só um gatilho para gastar 3 runs num braço treinado.
3. Reconciliar `CLAUDE.md`, o diagrama TikZ e o paper: a linha correta passa a ser
   **"2688→1024 (B1/B2); 2560→1024 (B0)"**, e o diagrama não pode mais mostrar a
   cross-attention.

In [20]:
final_metrics = trainer.evaluate()
for k, v in sorted(final_metrics.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

with open(RUN_DIR / "final_metrics.json", "w") as f:
    json.dump({k: (float(v) if isinstance(v, (int, float)) else v)
               for k, v in final_metrics.items()}, f, indent=2)
print(f"\n✓ métricas gravadas em {RUN_DIR / 'final_metrics.json'}")

2026-08-09 19:16:50,857 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'ICMNorm', 'FMeasure']
2026-08-09 19:16:50,900 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-09 19:16:51,023 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Normalized evaluation method
2026-08-09 19:16:51,025 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-09 19:16:51,159 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2026-08-09 19:16:51,277 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
2026-08-09 19:16:51,401 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']
2026-08-09 19:16:51,497 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft eva

Training Loss,Validation Loss,Epoch,Task A Icm Hard,Task A Icmnorm Hard,Task A Fmeasure Hard,Task A Icmsoft Soft,Task A Icmsoftnorm Soft,Task A Crossentropy Soft,Task B Icm Hard,Task B Icmnorm Hard,Task B Fmeasure Hard,Task B Icmsoft Soft,Task B Icmsoftnorm Soft,Task B Crossentropy Soft,Task C Icmsoft Soft,Task C Icmsoftnorm Soft,Task C Icm Hard,Task C Icmnorm Hard,Task C Fmeasure Hard,Proxy Task A F1,Proxy Task A Auc,Proxy Task B F1,Proxy Task C F1
0.711585,1.753174,15,0.107256,0.553640,0.701696,0.033199,0.505834,1.087417,-0.009159,0.496841,nan,-0.912985,0.394716,nan,-6.677875,0.151404,-0.646183,0.343594,nan,0.701696,0.780987,0.610427,0.333143


  eval_loss: 1.7532
  eval_proxy_task_a_auc: 0.7810
  eval_proxy_task_a_f1: 0.7017
  eval_proxy_task_b_f1: 0.6104
  eval_proxy_task_c_f1: 0.3331
  eval_task_a_crossentropy_soft: 1.0874
  eval_task_a_fmeasure_hard: 0.7017
  eval_task_a_icm_hard: 0.1073
  eval_task_a_icmnorm_hard: 0.5536
  eval_task_a_icmsoft_soft: 0.0332
  eval_task_a_icmsoftnorm_soft: 0.5058
  eval_task_b_crossentropy_soft: nan
  eval_task_b_fmeasure_hard: nan
  eval_task_b_icm_hard: -0.0092
  eval_task_b_icmnorm_hard: 0.4968
  eval_task_b_icmsoft_soft: -0.9130
  eval_task_b_icmsoftnorm_soft: 0.3947
  eval_task_c_fmeasure_hard: nan
  eval_task_c_icm_hard: -0.6462
  eval_task_c_icmnorm_hard: 0.3436
  eval_task_c_icmsoft_soft: -6.6779
  eval_task_c_icmsoftnorm_soft: 0.1514

✓ métricas gravadas em run_B2_bio_viewer_seed42_B15/final_metrics.json
